### **Atención moderna en Transformers y LLMs**

#### **De MHA a GQA, MLA, Sliding Window, Sparse Attention, Gated Attention e Hybrid Attention**

Este cuaderno es el complemento natural del cuaderno de **atención clásica**. Aquí ya no se estudia Seq2Seq con RNN como tema principal, sino cómo la atención de Transformers se modifica para resolver problemas actuales de inferencia:

- costo cuadrático de la atención completa,
- crecimiento del **KV cache**,
- contexto largo,
- atención local y dispersa,
- compresión de claves/valores,
- capas híbridas que combinan módulos baratos con capas de atención más fuertes.

La referencia conceptual principal es el artículo de Sebastian Raschka, *A Visual Guide to Attention Variants in Modern LLMs*.

> Nota importante: varias implementaciones son **didácticas**. Sirven para entender tensores, máscaras y costos. No pretenden reproducir kernels optimizados ni implementaciones industriales exactas de DeepSeek, Qwen, Kimi, Gemma o Nemotron.


#### **0. Preparación del entorno**

Usaremos PyTorch para implementar las variantes mínimas y Matplotlib para visualizar matrices de atención y máscaras. Todas las cadenas visibles y comentarios están en español.


In [ ]:
# Configuración común del cuaderno.
import math
import random
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

try:
    import pandas as pd
except Exception:
    pd = None

SEED = 1234
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo disponible: {device}")
print(f"Versión de PyTorch: {torch.__version__}")


In [ ]:
# Utilidades de visualización y verificación.
def mostrar_matriz(matriz: torch.Tensor, titulo: str, etiquetas: Optional[List[str]] = None) -> None:
    """Muestra una matriz 2D como mapa de calor simple."""
    matriz_cpu = matriz.detach().float().cpu()
    plt.figure(figsize=(5.2, 4.4))
    plt.imshow(matriz_cpu)
    plt.title(titulo)
    plt.xlabel("posición clave / token visible")
    plt.ylabel("posición consulta / token actualizado")
    plt.colorbar()
    if etiquetas is not None:
        plt.xticks(range(len(etiquetas)), etiquetas, rotation=45, ha="right")
        plt.yticks(range(len(etiquetas)), etiquetas)
    plt.tight_layout()
    plt.show()


def comprobar_suma_uno(pesos: torch.Tensor, mascara_valida: Optional[torch.Tensor] = None, atol: float = 1e-5) -> None:
    """Comprueba que las filas de atención válidas sumen aproximadamente 1."""
    suma = pesos.sum(dim=-1)
    if mascara_valida is not None:
        suma = suma[mascara_valida]
    assert torch.allclose(suma, torch.ones_like(suma), atol=atol), suma


def aplicar_mascara(scores: torch.Tensor, mask: Optional[torch.Tensor]) -> torch.Tensor:
    """Aplica una máscara booleana donde True significa posición prohibida."""
    if mask is None:
        return scores
    return scores.masked_fill(mask, torch.finfo(scores.dtype).min)


#### **1. Recordatorio: self-attention y scaled dot-product attention**

La técnica base sigue siendo:

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Las variantes modernas cambian principalmente **cómo se construyen, comparten, comprimen o restringen** `Q`, `K`, `V` y las máscaras de atención.


<figure style="text-align:center; margin: 1.2em 0;">
  <img src="https://substackcdn.com/image/fetch/$s_!VJyp!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F7dfb70d9-5eb6-44df-a880-3a37dfe72d7f_1836x1116.png" style="max-width:55%; border: 1px solid #ddd; border-radius: 8px;" />
  <figcaption style="font-size:0.9em; color:#555; margin-top:0.5em;">Un cabecera de atención ya constituye un mecanismo completo. Un conjunto de proyecciones aprendidas produce una matriz de atención y un flujo de salida sensible al contexto.Fuente visual: Raschka.</figcaption>
</figure>

In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Atención escalada por producto punto.

    Formas esperadas:
    - Q: [..., longitud_consulta, d_k]
    - K: [..., longitud_clave, d_k]
    - V: [..., longitud_clave, d_v]
    - mask: difundible a [..., longitud_consulta, longitud_clave]
      con True en posiciones prohibidas.
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    scores = aplicar_mascara(scores, mask)
    pesos = torch.softmax(scores, dim=-1)
    salida = pesos @ V
    return salida, pesos

# Ejemplo mínimo con una oración corta.
tokens = ["La", "vida", "es", "corta", "come", "postre", "primero"]
batch_size, seq_len, d_model = 1, len(tokens), 16
x = torch.randn(batch_size, seq_len, d_model, device=device)

salida, pesos = scaled_dot_product_attention(x, x, x)
print("Entrada:", tuple(x.shape))
print("Salida:", tuple(salida.shape))
print("Pesos:", tuple(pesos.shape))
comprobar_suma_uno(pesos)
mostrar_matriz(pesos[0], "Self-attention completa: pesos token-token", tokens)


#### **2. Máscara causal**

En un decoder autoregresivo, cada posición solo puede mirar tokens anteriores o el token actual. La parte superior derecha de la matriz se bloquea.


In [ ]:
def causal_mask(seq_len: int, device=None) -> torch.Tensor:
    """Máscara causal: True indica posiciones futuras prohibidas."""
    mascara = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1)
    return mascara.unsqueeze(0).unsqueeze(0)  # [1, 1, T, T]

mascara_causal = causal_mask(seq_len, device=device)
salida_causal, pesos_causales = scaled_dot_product_attention(
    x.unsqueeze(1), x.unsqueeze(1), x.unsqueeze(1), mask=mascara_causal
)
print("Salida causal:", tuple(salida_causal.shape))
mostrar_matriz((~mascara_causal[0, 0]).float(), "Posiciones visibles bajo máscara causal", tokens)
mostrar_matriz(pesos_causales[0, 0], "Pesos con máscara causal", tokens)


#### **3. Multi-Head Attention como punto de partida**

MHA replica la atención en varias cabeceras. Cada cabecera aprende proyecciones distintas y puede especializarse en relaciones diferentes. Esta es la base contra la cual compararemos GQA, MLA y las variantes de contexto largo.


<figure style="text-align:center; margin: 1.2em 0;">
  <img src="https://substackcdn.com/image/fetch/$s_!MSOX!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F5f47ffd2-bb3a-4936-9720-a88b8d867337_1766x1154.png" style="max-width:50%; border: 1px solid #ddd; border-radius: 8px;" />
  <figcaption style="font-size:0.9em; color:#555; margin-top:0.5em;">La atención multi-cabecera mantiene la misma estrategia básica de atención, pero la repite en varias cabeceras en paralelo para que el modelo pueda aprender varios patrones de token a token a la vez.Fuente visual: Raschka.</figcaption>
</figure>

In [ ]:
class MultiHeadAttentionScratch(nn.Module):
    """Multi-head attention didáctica con máscara opcional."""

    def __init__(self, d_model: int, num_heads: int, bias: bool = False):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=bias)
        self.Wk = nn.Linear(d_model, d_model, bias=bias)
        self.Wv = nn.Linear(d_model, d_model, bias=bias)
        self.out_proj = nn.Linear(d_model, d_model, bias=bias)

    def _separar_cabeceras(self, x: torch.Tensor) -> torch.Tensor:
        b, t, _ = x.shape
        return x.view(b, t, self.num_heads, self.head_dim).transpose(1, 2)

    def _unir_cabeceras(self, x: torch.Tensor) -> torch.Tensor:
        b, h, t, d = x.shape
        return x.transpose(1, 2).contiguous().view(b, t, h * d)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None):
        Q = self._separar_cabeceras(self.Wq(x))
        K = self._separar_cabeceras(self.Wk(x))
        V = self._separar_cabeceras(self.Wv(x))
        contexto, pesos = scaled_dot_product_attention(Q, K, V, mask=mask)
        salida = self.out_proj(self._unir_cabeceras(contexto))
        return salida, pesos

mha = MultiHeadAttentionScratch(d_model=32, num_heads=4).to(device)
x_mha = torch.randn(2, 8, 32, device=device)
out_mha, attn_mha = mha(x_mha, mask=causal_mask(8, device=device))
print("Salida MHA:", tuple(out_mha.shape))
print("Pesos MHA:", tuple(attn_mha.shape))
assert out_mha.shape == x_mha.shape
mostrar_matriz(attn_mha[0, 0], "MHA: pesos de la cabecera 0 con máscara causal")


#### **4. KV cache: por qué GQA y MLA importan**

Durante inferencia autoregresiva, el modelo no quiere recalcular `K` y `V` de todo el prefijo en cada token nuevo. Por eso guarda un **KV cache** por capa. El problema es que ese cache crece con:

$$
\text{batch} \times \text{capas} \times \text{longitud} \times \text{kv\_heads} \times \text{head\_dim} \times 2
$$

El factor `2` aparece porque se guardan claves y valores.


<figure style="text-align:center; margin: 1.2em 0;">
  <img src="https://substackcdn.com/image/fetch/$s_!BMEL!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F2d04f89a-aad5-4af7-8b46-4623d1d53ae9_5860x5682.webp" style="max-width:55%; border: 1px solid #ddd; border-radius: 8px;" />
  <figcaption style="font-size:0.9em; color:#555; margin-top:0.5em;">Tamaños totales de caché KV para Sarvam de 105B (usando MLA) frente a Sarvam de 30B (usando GQA), frente a usar MHA simple. Fuente visual: Raschka.</figcaption>
</figure>

In [ ]:
def kv_cache_gb(
    batch_size: int,
    num_layers: int,
    seq_len: int,
    num_kv_heads: int,
    head_dim: int,
    bytes_per_element: int = 2,
) -> float:
    """Estima el tamaño del KV cache en GiB."""
    elementos = batch_size * num_layers * seq_len * num_kv_heads * head_dim * 2
    return elementos * bytes_per_element / (1024 ** 3)

config = {
    "batch_size": 1,
    "num_layers": 32,
    "num_heads": 32,
    "head_dim": 128,
    "bytes_per_element": 2,  # bf16/fp16
}

filas = []
for seq in [4096, 8192, 32768, 131072]:
    filas.append({
        "contexto": seq,
        "MHA_32_kv_heads_GiB": kv_cache_gb(config["batch_size"], config["num_layers"], seq, 32, config["head_dim"]),
        "GQA_8_kv_heads_GiB": kv_cache_gb(config["batch_size"], config["num_layers"], seq, 8, config["head_dim"]),
        "MQA_1_kv_head_GiB": kv_cache_gb(config["batch_size"], config["num_layers"], seq, 1, config["head_dim"]),
    })

if pd is not None:
    tabla_kv = pd.DataFrame(filas)
    display(tabla_kv)
else:
    for fila in filas:
        print(fila)


In [ ]:
# Visualización del crecimiento del KV cache.
contextos = torch.tensor([4096, 8192, 32768, 131072])
mha_gb = [kv_cache_gb(1, 32, int(s), 32, 128) for s in contextos]
gqa_gb = [kv_cache_gb(1, 32, int(s), 8, 128) for s in contextos]
mqa_gb = [kv_cache_gb(1, 32, int(s), 1, 128) for s in contextos]

plt.figure(figsize=(6, 4))
plt.plot(contextos, mha_gb, marker="o", label="MHA: 32 KV heads")
plt.plot(contextos, gqa_gb, marker="o", label="GQA: 8 KV heads")
plt.plot(contextos, mqa_gb, marker="o", label="MQA: 1 KV head")
plt.xlabel("Longitud de contexto")
plt.ylabel("KV cache estimado (GiB)")
plt.title("Crecimiento del KV cache")
plt.legend()
plt.tight_layout()
plt.show()


#### **5. Grouped-Query Attention (GQA)**

GQA conserva muchas cabeceras de consulta, pero reduce el número de cabeceras de `K` y `V`. Varias query heads comparten las mismas claves y valores. Esto reduce memoria y tráfico del KV cache, especialmente en contexto largo.

MHA:

```text
num_heads = num_kv_heads
```

GQA:

```text
num_heads > num_kv_heads
```

MQA puede verse como el caso extremo:

```text
num_kv_heads = 1
```


In [ ]:
def repetir_kv(x: torch.Tensor, repeticiones: int) -> torch.Tensor:
    """Repite cabeceras K/V para alinearlas con el número de query heads.

    Entrada:  [batch, kv_heads, seq_len, head_dim]
    Salida:   [batch, kv_heads * repeticiones, seq_len, head_dim]

    Nota: en kernels eficientes se evita materializar copias innecesarias.
    Aquí lo hacemos por claridad didáctica.
    """
    b, kv_heads, t, d = x.shape
    x = x[:, :, None, :, :].expand(b, kv_heads, repeticiones, t, d)
    return x.reshape(b, kv_heads * repeticiones, t, d)

class GroupedQueryAttention(nn.Module):
    """GQA didáctica: más Q heads que K/V heads."""

    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int, bias: bool = False):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads"
        assert num_heads % num_kv_heads == 0, "num_heads debe ser múltiplo de num_kv_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_heads
        self.repeticiones = num_heads // num_kv_heads

        self.Wq = nn.Linear(d_model, num_heads * self.head_dim, bias=bias)
        self.Wk = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=bias)
        self.Wv = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=bias)
        self.out_proj = nn.Linear(d_model, d_model, bias=bias)

    def _proyectar(self, capa: nn.Linear, x: torch.Tensor, heads: int) -> torch.Tensor:
        b, t, _ = x.shape
        return capa(x).view(b, t, heads, self.head_dim).transpose(1, 2)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None):
        Q = self._proyectar(self.Wq, x, self.num_heads)
        K = self._proyectar(self.Wk, x, self.num_kv_heads)
        V = self._proyectar(self.Wv, x, self.num_kv_heads)
        K = repetir_kv(K, self.repeticiones)
        V = repetir_kv(V, self.repeticiones)
        contexto, pesos = scaled_dot_product_attention(Q, K, V, mask=mask)
        b, h, t, d = contexto.shape
        salida = contexto.transpose(1, 2).contiguous().view(b, t, h * d)
        return self.out_proj(salida), pesos

gqa = GroupedQueryAttention(d_model=64, num_heads=8, num_kv_heads=2).to(device)
x_gqa = torch.randn(2, 10, 64, device=device)
out_gqa, attn_gqa = gqa(x_gqa, mask=causal_mask(10, device=device))
print("Salida GQA:", tuple(out_gqa.shape))
print("Pesos GQA:", tuple(attn_gqa.shape))
assert out_gqa.shape == x_gqa.shape
mostrar_matriz(attn_gqa[0, 0], "GQA: cabecera de consulta 0")


<figure style="text-align:center; margin: 1.2em 0;">
  <img src="https://substackcdn.com/image/fetch/$s_!h6wM!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F6f39923c-8357-487d-9e69-40ee18a902e8_2523x1248.png" style="max-width:55%; border: 1px solid #ddd; border-radius: 8px;" />
  <figcaption style="font-size:0.9em; color:#555; margin-top:0.5em;">GQA mantiene el mismo patrón de atención general que MHA, pero reduce el número de encabezados clave-valor al compartirlos entre varias cabeceras de consulta. Fuente visual: Raschka.</figcaption>
</figure>

#### **6. Multi-Head Latent Attention (MLA) didáctica**

MLA busca reducir el KV cache de otra forma. En GQA se reducen las cabeceras K/V. En MLA se guarda una representación latente comprimida y desde ella se reconstruyen estados útiles para atención.

Esta implementación es una aproximación didáctica:

```text
x -> c_latente -> reconstrucción de K,V -> atención
```

No reproduce todos los detalles de DeepSeek MLA, pero sí muestra la diferencia esencial: **compresión del cache** en lugar de solo compartir K/V.


<figure style="text-align:center; margin: 1.2em 0;">
  <img src="https://substackcdn.com/image/fetch/$s_!FcJB!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Fe29cb535-8854-4412-af8d-9bca8d8d05f2_1550x858.webp" style="max-width:55%; border: 1px solid #ddd; border-radius: 8px;" />
  <figcaption style="font-size:0.9em; color:#555; margin-top:0.5em;">MLA reduce el cache guardando representaciones latentes comprimidas. Fuente visual: Raschka.</figcaption>
</figure>

In [ ]:
class ToyLatentKVAttention(nn.Module):
    """MLA didáctica: comprime x en un estado latente y reconstruye K/V."""

    def __init__(self, d_model: int, num_heads: int, latent_dim: int, bias: bool = False):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.latent_dim = latent_dim

        self.Wq = nn.Linear(d_model, d_model, bias=bias)
        self.compresor = nn.Linear(d_model, latent_dim, bias=bias)
        self.reconstruir_k = nn.Linear(latent_dim, d_model, bias=bias)
        self.reconstruir_v = nn.Linear(latent_dim, d_model, bias=bias)
        self.out_proj = nn.Linear(d_model, d_model, bias=bias)

    def _heads(self, x: torch.Tensor) -> torch.Tensor:
        b, t, _ = x.shape
        return x.view(b, t, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None):
        c_latente = self.compresor(x)
        Q = self._heads(self.Wq(x))
        K = self._heads(self.reconstruir_k(c_latente))
        V = self._heads(self.reconstruir_v(c_latente))
        contexto, pesos = scaled_dot_product_attention(Q, K, V, mask=mask)
        b, h, t, d = contexto.shape
        salida = contexto.transpose(1, 2).contiguous().view(b, t, h * d)
        return self.out_proj(salida), pesos, c_latente

mla_toy = ToyLatentKVAttention(d_model=64, num_heads=8, latent_dim=16).to(device)
x_mla = torch.randn(2, 12, 64, device=device)
out_mla, attn_mla, cache_latente = mla_toy(x_mla, mask=causal_mask(12, device=device))
print("Salida MLA didáctica:", tuple(out_mla.shape))
print("Cache latente:", tuple(cache_latente.shape))
assert out_mla.shape == x_mla.shape
mostrar_matriz(attn_mla[0, 0], "MLA didáctica: atención de la cabecera 0")


In [ ]:
def cache_latente_gb(batch_size: int, num_layers: int, seq_len: int, latent_dim: int, bytes_per_element: int = 2) -> float:
    """Estimación de cache latente tipo MLA en GiB.

    En esta versión didáctica solo guardamos un vector latente por token y capa.
    """
    elementos = batch_size * num_layers * seq_len * latent_dim
    return elementos * bytes_per_element / (1024 ** 3)

filas = []
for seq in [4096, 8192, 32768, 131072]:
    filas.append({
        "contexto": seq,
        "MHA_KV_GiB": kv_cache_gb(1, 32, seq, 32, 128),
        "GQA_8KV_GiB": kv_cache_gb(1, 32, seq, 8, 128),
        "MLA_latente_512_GiB": cache_latente_gb(1, 32, seq, 512),
        "MLA_latente_256_GiB": cache_latente_gb(1, 32, seq, 256),
    })

if pd is not None:
    display(pd.DataFrame(filas))
else:
    for fila in filas:
        print(fila)


#### **7. Sliding Window Attention (SWA)**

SWA es atención local causal. Cada token mira solo una ventana de tokens anteriores. Esto reduce el costo de capas locales en contextos largos.

La diferencia frente a la atención local clásica del cuaderno anterior es el contexto: aquí aparece dentro de arquitecturas de LLMs modernos y suele combinarse con capas globales ocasionales.


<figure style="text-align:center; margin: 1.2em 0;">
  <img src="https://substackcdn.com/image/fetch/$s_!edFZ!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F48a9cd31-24a5-47d9-ae37-506763ebc67d_1292x704.png" style="max-width:55%; border: 1px solid #ddd; border-radius: 8px;" />
  <figcaption style="font-size:0.9em; color:#555; margin-top:0.5em;"> La atención regular es atención global, mientras que la atención de ventana deslizante es atención local. La atención global permite que cada token vea el prefijo completo, SWA convierte muchas de esas capas en capas de atención local.Fuente visual: Raschka.</figcaption>
</figure>

In [ ]:
def sliding_window_causal_mask(seq_len: int, window_size: int, device=None) -> torch.Tensor:
    """Máscara SWA causal.

    True significa posición prohibida.
    Un token i puede atender a j si:
      - j <= i, causalidad;
      - i - j <= window_size, ventana local.
    """
    i = torch.arange(seq_len, device=device)[:, None]
    j = torch.arange(seq_len, device=device)[None, :]
    futuro = j > i
    fuera_ventana = (i - j) > window_size
    return (futuro | fuera_ventana).unsqueeze(0).unsqueeze(0)

for ventana in [1, 2, 4]:
    mask_swa = sliding_window_causal_mask(seq_len=10, window_size=ventana, device=device)
    mostrar_matriz((~mask_swa[0, 0]).float(), f"SWA: posiciones visibles con ventana={ventana}")


In [ ]:
x_swa = torch.randn(1, 10, 32, device=device)
swa_mha = MultiHeadAttentionScratch(d_model=32, num_heads=4).to(device)
mask_swa = sliding_window_causal_mask(seq_len=10, window_size=3, device=device)
out_swa, attn_swa = swa_mha(x_swa, mask=mask_swa)
print("Salida SWA usando MHA con máscara local:", tuple(out_swa.shape))
mostrar_matriz(attn_swa[0, 0], "Pesos de atención bajo SWA, cabecera 0")


#### **8. Sparse / top-k attention: aproximación didáctica a DSA**

DeepSeek Sparse Attention no usa una ventana fija como SWA. El patrón disperso se aprende mediante un indexer y un selector. Aquí implementamos una versión **top-k simple**:

```text
scores QKᵀ -> conservar los k scores más altos -> enmascarar el resto -> softmax
```

Esto enseña la idea de seleccionar un subconjunto, pero no reproduce la arquitectura completa de DSA.


<figure style="text-align:center; margin: 1.2em 0;">
  <img src="https://substackcdn.com/image/fetch/$s_!TvQp!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Fed46e1ba-6c43-421a-b98a-973987cff3f7_4065x4327.webp" style="max-width:55%; border: 1px solid #ddd; border-radius: 8px;" />
  <figcaption style="font-size:0.9em; color:#555; margin-top:0.5em;"> De forma similar a la atención de ventana deslizante, DeepSeek Sparse Attention también restringe cada token a un subconjunto de tokens anteriores, pero no lo hace con una ventana local fija. Fuente visual: Raschka.</figcaption>
</figure>

In [ ]:
def topk_sparse_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    k: int,
    mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Atención top-k didáctica.

    Devuelve salida, pesos y máscara dispersa final.
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    scores = aplicar_mascara(scores, mask)

    k = min(k, scores.size(-1))
    valores_topk, indices_topk = torch.topk(scores, k=k, dim=-1)
    mascara_dispersa = torch.ones_like(scores, dtype=torch.bool)
    mascara_dispersa.scatter_(-1, indices_topk, False)

    scores_sparse = scores.masked_fill(mascara_dispersa, torch.finfo(scores.dtype).min)
    pesos = torch.softmax(scores_sparse, dim=-1)
    salida = pesos @ V
    return salida, pesos, mascara_dispersa

x_sparse = torch.randn(1, 1, 12, 16, device=device)  # [batch, heads, seq, dim]
mask_causal_12 = causal_mask(12, device=device)
out_sparse, pesos_sparse, mascara_sparse = topk_sparse_attention(
    x_sparse, x_sparse, x_sparse, k=3, mask=mask_causal_12
)
print("Salida sparse:", tuple(out_sparse.shape))
mostrar_matriz((~mascara_sparse[0, 0]).float(), "Patrón visible top-k sparse, cabecera 0")
mostrar_matriz(pesos_sparse[0, 0], "Pesos top-k sparse, cabecera 0")


In [ ]:
class ToyLearnedSparseSelector(nn.Module):
    """Selector aprendido simplificado: produce scores y conserva top-k.

    No es DSA real. Solo separa pedagógicamente indexer y selector.
    """

    def __init__(self, d_model: int):
        super().__init__()
        self.indexer_q = nn.Linear(d_model, d_model, bias=False)
        self.indexer_k = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor, k: int, mask: Optional[torch.Tensor] = None):
        Q_idx = self.indexer_q(x)
        K_idx = self.indexer_k(x)
        scores = Q_idx @ K_idx.transpose(-2, -1) / math.sqrt(x.size(-1))
        scores = aplicar_mascara(scores, mask.squeeze(1) if mask is not None else None)
        _, indices = torch.topk(scores, k=min(k, scores.size(-1)), dim=-1)
        seleccion = torch.ones_like(scores, dtype=torch.bool)
        seleccion.scatter_(-1, indices, False)
        return seleccion, scores

selector = ToyLearnedSparseSelector(d_model=16).to(device)
x_tokens = torch.randn(1, 12, 16, device=device)
seleccion, scores_idx = selector(x_tokens, k=3, mask=mask_causal_12)
mostrar_matriz((~seleccion[0]).float(), "Selector aprendido didáctico: posiciones conservadas")


#### **9. Gated Attention con QK-Norm y partial RoPE**

Gated Attention no es una familia completamente separada. Es atención completa modificada con mecanismos de control y estabilidad:

1. una compuerta sobre la salida de atención;
2. normalización de `Q` y `K`;
3. aplicación parcial de RoPE en algunas dimensiones.

La implementación siguiente integra los tres elementos de forma didáctica.


<figure style="text-align:center; margin: 1.2em 0;">
  <img src="https://substackcdn.com/image/fetch/$s_!syqZ!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F2d109b38-96a1-4830-b1f0-7274e405317a_3973x5350.webp" style="max-width:45%; border: 1px solid #ddd; border-radius: 8px;" />
  <figcaption style="font-size:0.9em; color:#555; margin-top:0.5em;"> Gate aparece después de la salida de atención del producto escalar escalada y antes de la proyección de salida en una arquitectura de contexto largo diferente.
Fuente visual: Raschka.</figcaption>
</figure>

In [ ]:
def zero_centered_qk_norm(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Normalización simplificada: centra y escala cada vector."""
    x = x - x.mean(dim=-1, keepdim=True)
    return x / torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)


def aplicar_rope_parcial(x: torch.Tensor, fraction: float = 0.5) -> torch.Tensor:
    """Aplica una versión simple de RoPE a una fracción inicial de dimensiones.

    Forma: [batch, heads, seq_len, head_dim]
    """
    b, h, t, d = x.shape
    d_rope = int(d * fraction)
    d_rope = d_rope - (d_rope % 2)
    if d_rope <= 0:
        return x

    x_rope = x[..., :d_rope]
    x_rest = x[..., d_rope:]

    posiciones = torch.arange(t, device=x.device, dtype=x.dtype)
    freqs = 1.0 / (10000 ** (torch.arange(0, d_rope, 2, device=x.device, dtype=x.dtype) / d_rope))
    angulos = posiciones[:, None] * freqs[None, :]
    cos = torch.cos(angulos)[None, None, :, :]
    sin = torch.sin(angulos)[None, None, :, :]

    x_even = x_rope[..., 0::2]
    x_odd = x_rope[..., 1::2]
    rot_even = x_even * cos - x_odd * sin
    rot_odd = x_even * sin + x_odd * cos
    x_rot = torch.stack([rot_even, rot_odd], dim=-1).flatten(-2)
    return torch.cat([x_rot, x_rest], dim=-1)

class GatedAttentionWithQKNormPartialRoPE(nn.Module):
    """Atención completa con compuerta, QK-Norm y partial RoPE."""

    def __init__(self, d_model: int, num_heads: int, rope_fraction: float = 0.5):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.rope_fraction = rope_fraction
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.gate = nn.Linear(d_model, d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def _heads(self, x: torch.Tensor) -> torch.Tensor:
        b, t, _ = x.shape
        return x.view(b, t, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None):
        Q = self._heads(self.Wq(x))
        K = self._heads(self.Wk(x))
        V = self._heads(self.Wv(x))

        Q = aplicar_rope_parcial(zero_centered_qk_norm(Q), self.rope_fraction)
        K = aplicar_rope_parcial(zero_centered_qk_norm(K), self.rope_fraction)

        contexto, pesos = scaled_dot_product_attention(Q, K, V, mask=mask)
        b, h, t, d = contexto.shape
        contexto = contexto.transpose(1, 2).contiguous().view(b, t, h * d)

        compuerta = torch.sigmoid(self.gate(x))
        salida = self.out_proj(compuerta * contexto)
        return salida, pesos, compuerta

gated = GatedAttentionWithQKNormPartialRoPE(d_model=64, num_heads=8, rope_fraction=0.5).to(device)
x_gated = torch.randn(2, 12, 64, device=device)
out_gated, attn_gated, gate_values = gated(x_gated, mask=causal_mask(12, device=device))
print("Salida gated attention:", tuple(out_gated.shape))
print("Compuerta:", tuple(gate_values.shape), "mín/max:", float(gate_values.min()), float(gate_values.max()))
assert out_gated.shape == x_gated.shape
mostrar_matriz(attn_gated[0, 0], "Gated attention: pesos de la cabecera 0")


#### **10. Hybrid Attention**

Hybrid Attention no significa simplemente combinar SWA con GQA o MLA con sparse attention. En el artículo, la idea es más arquitectónica:

```text
mantener una pila tipo Transformer,
pero reemplazar la mayoría de capas de full attention
por módulos lineales, recurrentes o de espacio de estados,
y conservar algunas capas pesadas para recuperación precisa.
```

Ejemplo de patrón 3:1:

```text
DeltaNet / módulo barato
DeltaNet / módulo barato
DeltaNet / módulo barato
Gated Attention / capa pesada
```


<figure style="text-align:center; margin: 1.2em 0;">
  <img src="https://substackcdn.com/image/fetch/$s_!XRZY!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F876009cb-5479-4e95-9b7d-faebe1f87e89_3252x2158.webp" style="max-width:55%; border: 1px solid #ddd; border-radius: 8px;" />
  <figcaption style="font-size:0.9em; color:#555; margin-top:0.5em;"> El patrón híbrido básico, donde la mayoría de los bloques son mezcladores de secuencia menos costoso y cada cuarto bloque restaura una capa de atención más pesada. Fuente visual: Raschka.</figcaption>
</figure>

In [ ]:
def patron_hibrido(num_layers: int = 16, cheap_per_full: int = 3) -> List[str]:
    """Genera un patrón híbrido: N capas baratas por cada capa pesada."""
    bloques = []
    ciclo = cheap_per_full + 1
    for i in range(num_layers):
        if (i + 1) % ciclo == 0:
            bloques.append("gated_full_attention")
        else:
            bloques.append("linear_or_ssm_mixer")
    return bloques

bloques = patron_hibrido(num_layers=16, cheap_per_full=3)
for i, bloque in enumerate(bloques, start=1):
    print(f"Capa {i:02d}: {bloque}")


In [ ]:
def costo_relativo_full_attention(seq_len: int, d_model: int) -> float:
    """Costo relativo simplificado O(T^2 d)."""
    return float(seq_len * seq_len * d_model)


def costo_relativo_modulo_barato(seq_len: int, d_model: int) -> float:
    """Costo relativo simplificado O(T d^2) o O(T d), modelado como lineal en T para comparar."""
    return float(seq_len * d_model * d_model)


def costo_stack_hibrido(seq_len: int, d_model: int, bloques: List[str]) -> float:
    costo = 0.0
    for bloque in bloques:
        if bloque == "gated_full_attention":
            costo += costo_relativo_full_attention(seq_len, d_model)
        else:
            costo += costo_relativo_modulo_barato(seq_len, d_model)
    return costo

seqs = [1024, 4096, 8192, 32768]
d_model_demo = 1024
bloques_full = ["gated_full_attention"] * 16
bloques_hibridos = patron_hibrido(16, cheap_per_full=3)

costos_full = [costo_stack_hibrido(s, d_model_demo, bloques_full) for s in seqs]
costos_hybrid = [costo_stack_hibrido(s, d_model_demo, bloques_hibridos) for s in seqs]

plt.figure(figsize=(6, 4))
plt.plot(seqs, costos_full, marker="o", label="16 capas full attention")
plt.plot(seqs, costos_hybrid, marker="o", label="patrón híbrido 3:1")
plt.xlabel("Longitud de contexto")
plt.ylabel("Costo relativo simplificado")
plt.title("Motivación de arquitecturas híbridas")
plt.legend()
plt.tight_layout()
plt.show()


#### **11. Tabla final de comparación**

Esta tabla resume el propósito pedagógico de cada variante.


In [ ]:

resumen = [
    {
        "variante": "MHA",
        "idea": "Cada cabecera tiene sus propias Q/K/V",
        "optimiza": "calidad y diversidad de patrones",
        "costo_cache": "alto",
        "implementación_en_cuaderno": "módulo completo didáctico",
    },
    {
        "variante": "GQA",
        "idea": "Muchas Q heads comparten menos K/V heads",
        "optimiza": "KV cache y tráfico de memoria",
        "costo_cache": "medio/bajo",
        "implementación_en_cuaderno": "correcta como didáctica; no kernel eficiente",
    },
    {
        "variante": "MLA",
        "idea": "Guardar estado latente comprimido",
        "optimiza": "representación del cache",
        "costo_cache": "bajo",
        "implementación_en_cuaderno": "toy, no DeepSeek real",
    },
    {
        "variante": "SWA",
        "idea": "Atención local causal por ventana",
        "optimiza": "contexto largo en capas locales",
        "costo_cache": "depende del patrón local/global",
        "implementación_en_cuaderno": "máscara local causal correcta",
    },
    {
        "variante": "Sparse/top-k",
        "idea": "Conservar subconjunto de posiciones relevantes",
        "optimiza": "revisitar menos tokens previos",
        "costo_cache": "depende de k y del indexer",
        "implementación_en_cuaderno": "top-k didáctico + selector aprendido toy",
    },
    {
        "variante": "Gated Attention",
        "idea": "Full attention con compuerta, QK-Norm y partial RoPE",
        "optimiza": "control y estabilidad",
        "costo_cache": "similar a full attention",
        "implementación_en_cuaderno": "bloque integrado didáctico",
    },
    {
        "variante": "Hybrid Attention",
        "idea": "Muchos módulos baratos + algunas capas de atención pesada",
        "optimiza": "contexto largo y eficiencia",
        "costo_cache": "menor que full stack",
        "implementación_en_cuaderno": "patrón de capas y costo relativo",
    },
]

if pd is not None:
    display(pd.DataFrame(resumen))
else:
    for fila in resumen:
        print(fila)


#### **12. Ejercicios propuestos**

1. Cambia `num_kv_heads` en GQA de `8 -> 4 -> 2 -> 1` y mida la reducción estimada de KV cache.
2. Compara SWA con ventanas `128`, `512`, `1024` y `4096` para un contexto de `32768` tokens.
3. Reemplaza el top-k sparse por una máscara que combine ventana local + tokens globales fijos.
4. Modifica `rope_fraction` en Gated Attention y observe si las formas de los tensores se mantienen.
5. Diseña un patrón híbrido `5:1` y compárelo contra el patrón `3:1`.
6. Explica por qué GQA y SWA se pueden combinar: una técnica reduce K/V por token y la otra reduce el rango atendido por capa.


In [ ]:
## Tus respuestas